In [1]:
# ============================================================
# Cell 1: load data
# ============================================================
import re
import pandas as pd
import os
from collections import defaultdict
from difflib import SequenceMatcher

notebook_path = os.getcwd()
in_dir_notifications_raw = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "notification_historical", "notification.jsonl"))
in_dir_master_direction   = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "master_directory", "master_directory.jsonl"))

circulars         = pd.read_json(in_dir_notifications_raw, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines=True)
print(f"circulars: {len(circulars)}, master_directions: {len(master_directions)}")


# ============================================================
# Cell 2: citation regex + normalizer
# ============================================================
DOC_TYPE = r"(?:Directions?|Guidelines?|Regulations?|Rules?|Circulars?|Framework|Scheme)"

p1 = re.compile(
    r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)"
    r"(?:\s*\([^)]*\))*"
    r"\s*(?:[A-Za-z]+\s+){0,3}" + DOC_TYPE,
    re.IGNORECASE
)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–", "-").replace("—", "-")).strip(" ,-").lower()

def extract_names(text):
    if not isinstance(text, str):
        return []
    return [norm(x) for x in (p1.findall(text) + p2.findall(text))]


# ============================================================
# Cell 3: title + full-text + "lead paragraph" extraction
# ============================================================
# Lead cutoff (start of numbered paragraph "2.") is ONLY for the "does this
# doc even carry a citation" bucketing check further down. Matching itself
# always runs on found_text (full text) -- restricting matching to the lead
# was a real bug that cut real matches from 102 to 79.
lead_break = re.compile(r"\n\s*2\.\s")

def get_lead(text):
    if not isinstance(text, str):
        return ""
    m = lead_break.search(text)
    return text[:m.start()] if m else text[:600]

circulars["found_title"]     = circulars["title"].apply(extract_names)                        # title citations
circulars["found_text"]      = circulars["text"].apply(extract_names)                         # full text -- used for matching
circulars["found_text_lead"] = circulars["text"].apply(lambda t: extract_names(get_lead(t)))  # lead only -- used for bucketing


# ============================================================
# Cell 4: build master_lookup, flagging collisions instead of silently overwriting
# ============================================================
master_directions["extracted_names"] = master_directions["title"].apply(extract_names)

name_to_ids = defaultdict(set)
for _, r in master_directions.iterrows():
    for name in r["extracted_names"]:
        name_to_ids[name].add(r["id"])

collisions = {k: v for k, v in name_to_ids.items() if len(v) > 1}
print(f"{len(collisions)} names collide across master_directions (excluded from lookup):")
print(collisions)

unlinkable = master_directions[master_directions["extracted_names"].str.len() == 0]
print(f"{len(unlinkable)}/{len(master_directions)} master directions have no extractable name (can never be matched to)")

master_lookup = {name: next(iter(ids)) for name, ids in name_to_ids.items() if len(ids) == 1}
master_keys = list(master_lookup.keys())


# ============================================================
# Cell 5: title/full-text matching
# ============================================================
def match_row(row):
    for n in row["found_title"]:
        if n in master_lookup:
            return master_lookup[n], "title_match"
    for n in row["found_text"]:          # full text, not lead
        if n in master_lookup:
            return master_lookup[n], "text_match"
    return None, "no_match"

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(match_row(r)), axis=1)
print(circulars["match_method"].value_counts())

recovered_by_full_text = circulars[
    (circulars["match_method"] != "no_match") &
    (circulars["found_title"].str.len() == 0) &
    (~circulars["found_text_lead"].apply(lambda lst: any(n in master_lookup for n in lst)))
]
print(f"{len(recovered_by_full_text)} matches only found past the lead cutoff")
recovered_by_full_text[["id", "title", "matched_id"]]


# ============================================================
# Cell 6: fuzzy candidates (char-diff, not % similarity -- generic RBI phrases
# get reused verbatim across NBFC/SFB/AIFI titles, so ratio similarity flags
# false positives; real near-dupes differ by 1-3 chars, false ones by 20+)
# ============================================================
def char_diff(a, b):
    diff = 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes():
        if tag != "equal":
            diff += max(i2 - i1, j2 - j1)
    return diff

fuzzy_candidates = []
for idx, row in circulars[circulars["match_method"] == "no_match"].iterrows():
    for n in (row["found_title"] + row["found_text_lead"]):
        for key in master_keys:
            if char_diff(n, key) <= 5:   # small typo/formatting gap only, not a different phrase
                fuzzy_candidates.append((idx, row["id"], n, key))
                break

fuzzy_df = pd.DataFrame(fuzzy_candidates, columns=["row_idx", "circular_id", "extracted_name", "closest_master_name"])
print(f"{len(fuzzy_df)} candidates")
fuzzy_df


# ============================================================
# Cell 6b: apply fuzzy matches -- only where a row has ONE unambiguous
# candidate id. A row fuzzy-matching two different master ids gets flagged
# for review instead of silently keeping whichever came last.
# ============================================================
dupe_check = fuzzy_df.groupby("row_idx")["closest_master_name"].apply(
    lambda names: len({master_lookup[n] for n in names})
)
ambiguous_rows = dupe_check[dupe_check > 1].index
if len(ambiguous_rows):
    print(f"{len(ambiguous_rows)} rows have conflicting fuzzy candidates -- review before applying:")
    print(fuzzy_df[fuzzy_df["row_idx"].isin(ambiguous_rows)])

safe_fuzzy = fuzzy_df[~fuzzy_df["row_idx"].isin(ambiguous_rows)].drop_duplicates("row_idx")
for _, r in safe_fuzzy.iterrows():
    circulars.loc[r["row_idx"], "matched_id"]   = master_lookup[r["closest_master_name"]]
    circulars.loc[r["row_idx"], "match_method"] = "fuzzy_match"

print(circulars["match_method"].value_counts())


# ============================================================
# Cell 7: NBFC relevance (broadened to sibling entity types RBI regulates
# under the same umbrella) + discard/general split via entity taxonomy
# ============================================================
nbfc_pattern = re.compile(
    r"non[\s-]?banking financial compan|nbfc"
    r"|core investment compan(?:y|ies)|\bcic\b"
    r"|standalone primary dealer|\bspd\b"
    r"|mortgage guarantee compan(?:y|ies)|\bmgc\b"
    r"|non-?operative financial holding compan(?:y|ies)|\bnofhc\b"
    r"|housing finance compan(?:y|ies)|\bhfc\b",
    re.IGNORECASE
)
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, na=False) |
    circulars["title"].str.contains(nbfc_pattern, na=False)
)

# Entities RBI regulates that are explicitly NOT NBFC-family. A title naming
# one of these as its scope is a deliberate discard signal -- stronger than
# just "NBFC wasn't mentioned." Checked on title only (that's where scope is
# declared); expand this list against your actual master_directions titles.
other_entity_pattern = re.compile(
    r"regional rural bank"
    r"|urban co-?operative bank"
    r"|state co-?operative bank"
    r"|district central co-?operative bank"
    r"|scheduled commercial bank"
    r"|commercial bank"
    r"|payments? bank"
    r"|small finance bank"
    r"|local area bank"
    r"|co-?operative bank",
    re.IGNORECASE
)
circulars["names_other_entity"] = circulars["title"].str.contains(other_entity_pattern, na=False)

def bucket(row):
    if row["match_method"] != "no_match":
        return "linked"
    if row["is_nbfc_relevant"]:
        has_citation = len(row["found_title"]) > 0 or len(row["found_text_lead"]) > 0
        return "nbfc_citation_unmatched" if has_citation else "nbfc_standalone"
    # is_nbfc_relevant already covers "names both NBFC and another entity" --
    # that branch returns above, so reaching here means NBFC was NOT named.
    if row["names_other_entity"]:
        return "discard"
    return "general"   # genuinely ambiguous -- hand to the model

circulars["bucket"] = circulars.apply(bucket, axis=1)
print(circulars["bucket"].value_counts())


# ============================================================
# Cell 8: subject_code as an independent cross-check (not a primary matcher)
# ============================================================
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")
circulars["subject_code"]         = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

print(f"subject_code coverage -- circulars: {circulars['subject_code'].notna().mean():.0%}, "
      f"master_directions: {master_directions['subject_code'].notna().mean():.0%}")

# auto-drop any code shared by more than one master direction (generic/bucket
# codes, e.g. "33-01-010" was shared across 30 unrelated master directions)
code_counts = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
generic_codes = code_counts[code_counts > 1].index.tolist()
clean_master = master_directions[~master_directions["subject_code"].isin(generic_codes)]

code_matches = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)
both = code_matches[(code_matches["match_method"] != "no_match") & code_matches["id_master"].notna()]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between regex-match and code-match")


# ============================================================
# Cell 9: final tally
# ============================================================
print(circulars["bucket"].value_counts())
nbfc_total = circulars["bucket"].isin(["linked", "nbfc_citation_unmatched", "nbfc_standalone"]).sum()
print(f"NBFC-relevant: {nbfc_total} / {len(circulars)}")
print(f"Discard (named a different specific entity): {(circulars['bucket']=='discard').sum()} / {len(circulars)}")
print(f"General (ambiguous, needs model review): {(circulars['bucket']=='general').sum()} / {len(circulars)}")


# ============================================================
# Cell 10: spot-check samples -- closes the open item of never actually
# validating discard/general error rate. Change seed to pull a fresh sample.
# ============================================================
def spot_check(bucket_name, n=15, seed=0):
    pool = circulars[circulars["bucket"] == bucket_name]
    return pool.sample(min(n, len(pool)), random_state=seed)[["id", "title", "is_nbfc_relevant", "names_other_entity"]]

spot_check("discard")                  # confirm none quietly carry an NBFC clause in the body
spot_check("general")                  # confirm these are genuinely ambiguous, not a missed keyword
spot_check("nbfc_citation_unmatched")  # confirm the citation really isn't in the master list

circulars: 766, master_directions: 44
1 names collide across master_directions (excluded from lookup):
{'non-banking financial companies - miscellaneous': {13586, 12931}}
0/44 master directions have no extractable name (can never be matched to)
match_method
no_match       665
title_match     76
text_match      25
Name: count, dtype: int64
1 matches only found past the lead cutoff
3 candidates
match_method
no_match       663
title_match     76
text_match      25
fuzzy_match      2
Name: count, dtype: int64
bucket
discard                    395
nbfc_citation_unmatched    154
general                    113
linked                     103
nbfc_standalone              1
Name: count, dtype: int64
subject_code coverage -- circulars: 49%, master_directions: 75%
0 / 7 disagree between regex-match and code-match
bucket
discard                    395
nbfc_citation_unmatched    154
general                    113
linked                     103
nbfc_standalone              1
Name: count, dtype: int64

,id,title,is_nbfc_relevant,names_other_entity
87,13012,Reserve Bank of India (Urban Co-operative Bank...,True,True
222,13147,Reserve Bank of India (Commercial Banks – Asse...,True,True
751,13676,"Implementation of Section 51A of UAPA, 1967: U...",True,False
260,13185,Reserve Bank of India (Rural Co-operative Bank...,True,True
77,13002,Reserve Bank of India (Rural Co-operative Bank...,True,True
47,12972,Reserve Bank of India (All India Financial Ins...,True,False
127,13052,Reserve Bank of India (Regional Rural Banks – ...,True,True
258,13183,Reserve Bank of India (Regional Rural Banks – ...,True,True
150,13075,Reserve Bank of India (Local Area Banks – Cred...,True,True
351,13276,Reserve Bank of India (Credit Information Comp...,True,False


In [4]:
pd.set_option('display.max_colwidth', None)
def review_sample(method, n=15, seed=0):
    pool = circulars[circulars["match_method"] == method]
    sample = pool.sample(min(n, len(pool)), random_state=seed).copy()
    return sample.merge(
        master_directions[["id", "title"]].rename(columns={"title": "master_title"}),
        left_on="matched_id", right_on="id", suffixes=("", "_m")
    )[["id", "title", "master_title", "match_method"]]

review_sample("title_match")
review_sample("text_match")
review_sample("fuzzy_match")   # review ALL of these, not just a sample, if the count is small

,id,title,master_title,match_method
0,13379,"Reserve Bank of India (Non-Banking Financial Companies– Undertaking of Financial Services) –Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Undertaking of Financial Services) Directions, 2025 (Updated as on July 01, 2026)",fuzzy_match
1,13213,"Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025","Reserve Bank of India (Non-Operative Financial Holding Companies) Directions, 2025 (Updated as on December 05, 2025)",fuzzy_match


In [5]:
code_lookup = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code", "title"]],
    on="subject_code", how="left", suffixes=("", "_code_master")
)

recoverable = code_lookup[
    (code_lookup["match_method"] == "no_match") &
    code_lookup["id_code_master"].notna()
]
print(f"{len(recoverable)} unmatched circulars have a subject_code hit against a master direction")
recoverable[["id", "title", "title_code_master"]]

21 unmatched circulars have a subject_code hit against a master direction


,id,title,title_code_master
58,12983,"Reserve Bank of India (All India Financial Institutions - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on June 16, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
130,13055,"Reserve Bank of India (Regional Rural Banks - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on June 16, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
155,13080,"Reserve Bank of India (Local Area Banks – Prudential Norms on Capital Adequacy) Directions, 2025","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
172,13097,"Reserve Bank of India (Payments Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (updated as on May 08, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
202,13127,"Reserve Bank of India (Small Finance Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
234,13159,"Reserve Bank of India (Commercial Banks - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
275,13200,"Reserve Bank of India (Commercial Banks - Prudential Norms on Capital Adequacy) Amendment Directions, 2025","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
276,13201,"Reserve Bank of India (Small Finance Banks – Prudential Norms on Capital Adequacy) Amendment Directions, 2025","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
340,13265,"Reserve Bank of India (Commercial Banks - Prudential Norms on Capital Adequacy) Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
341,13266,"Reserve Bank of India (Small Finance Banks - Prudential Norms on Capital Adequacy) Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"


In [6]:
matched_ids = set(circulars.loc[circulars["match_method"] != "no_match", "matched_id"])
orphans = master_directions[~master_directions["id"].isin(matched_ids)]
nbfc_orphans = orphans[orphans["title"].str.contains(nbfc_pattern, na=False)]
print(f"{len(orphans)}/{len(master_directions)} master directions have zero linked circulars")
print(f"{len(nbfc_orphans)} of those are NBFC-titled")
nbfc_orphans[["id", "title"]]

2/44 master directions have zero linked circulars
2 of those are NBFC-titled


,id,title
0,12931,"Reserve Bank of India (Non-Banking Financial Companies – Miscellaneous) Directions, 2025 (Updated as on February 26, 2026)"
36,13586,"Reserve Bank of India (Non-Banking Financial Companies – Miscellaneous) Supervisory Directions, 2026"


In [7]:
# is_nbfc_relevant checks text too, so nothing in discard should contain the keyword
assert circulars.loc[circulars["bucket"] == "discard", "is_nbfc_relevant"].sum() == 0

# "general" should mean neither pattern hit - confirm no overlap
general_check = circulars[circulars["bucket"] == "general"]
assert general_check["is_nbfc_relevant"].sum() == 0
assert general_check["names_other_entity"].sum() == 0

In [8]:
# fill in ~15-20 pairs you've manually verified, including the Concentration
# Risk case you already confirmed
gold_pairs = {
    "circular_id_1": "master_id_1",
    "circular_id_2": "master_id_2",
}

for cid, expected in gold_pairs.items():
    row = circulars.loc[circulars["id"] == cid, "matched_id"]
    got = row.values[0] if len(row) else None
    status = "OK" if got == expected else f"MISMATCH (expected {expected}, got {got})"
    print(cid, status)

circular_id_1 MISMATCH (expected master_id_1, got None)
circular_id_2 MISMATCH (expected master_id_2, got None)


In [ ]:
Here's the throughline of just the linking/discarding work, stripped of the RAG/watcher/model architecture stuff:

**1. The core matching method**
RBI circulars follow a pattern: `"Reserve Bank of India (ENTITY – TOPIC) Directions/Guidelines/..."`. The approach extracts whatever's inside those parentheses (from both the circular's title and its text), normalizes it (lowercase, unify dashes, trim punctuation), and looks it up against the same extraction run on every master direction's title. Match → linked, with `title_match` vs `text_match` telling you where it was found. Verified this against your Concentration Risk example — worked immediately, title alone was enough.

**2. Problems found in that method**
- `master_lookup` was built by just overwriting a dict entry per name — if two master directions ever normalized to the same name, one would silently vanish. Found exactly one real case: `"non-banking financial companies - miscellaneous"` collided across two master direction ids.
- No check existed for master directions whose titles don't extract *any* name — those can never be matched to, silently.
- The "only search the first 600 characters of text" heuristic (meant to avoid picking up an unrelated citation buried deep in a long document) was arbitrary. Replaced with cutting at RBI's own structural boundary — the start of numbered paragraph "2." — since that's where the background/preamble reliably ends in their documents.

**3. Subject codes as a second, independent method**
Circulars and master directions both carry codes like `DOR.CRE.REC.210/07-03-008/2026-27`. The middle segment (`07-03-008`) looked like a topic identifier that persists across a document's amendments. Tested it:
- Only ~49% of circulars and ~75% of master directions even have this code.
- One code, `33-01-010`, turned out to be shared across 30 unrelated master directions — a generic bucket stamp, not a real identifier. Not filtering it out caused a merge explosion (one circular fanned out into 30 fake "matches").
- Once filtered, it added *zero* new real matches — it turned out the only circulars where a code-match was even possible had already been caught by the regex method. Its real value ended up being a trustworthy cross-check (0 disagreements against title/text matching), not a discovery tool.

**4. Fuzzy matching for near-misses**
Applied to the citations that didn't resolve — trying to catch things like "Company" vs "Companies." First attempt used percentage similarity (`difflib.get_close_matches`, 0.85 cutoff) and produced mostly false positives: RBI reuses identical generic topic phrases (e.g. "Classification, Valuation and Operation of Investment Portfolio") across totally different entity types — NBFC, Small Finance Banks, All India Financial Institutions — so a long shared suffix made unrelated documents look 85%+ similar. Fixed by switching from percentage similarity to raw character-difference count: real near-duplicates differed by 1-3 characters, false positives by 20+ — a clean, obvious gap once measured the right way.

**5. A bug I introduced and we caught**
When rebuilding the pipeline, I accidentally applied the "lead text only" restriction to the actual matching step too, not just the relevance-flagging step it was meant for — dropped total matches from 102 to 79. Fixed by restoring full-text search for matching (it already takes the first citation found, so it doesn't need the lead restriction) and keeping the lead-cutoff only for deciding whether a document "has a citation" for bucketing purposes.

**6. Where the numbers landed**
`title_match: 76, text_match: 25, fuzzy_match: 2` → **linked: 103**, `nbfc_citation_unmatched: 133`, `nbfc_standalone: 1`, `not_nbfc: 529`. NBFC-relevant total: **237/766**. Cross-checked clean against subject codes (0/7 disagreements).

**7. Still-open items from that point**
- The NBFC keyword check (`"nbfc"` / `"non-banking financial compan"`) was too narrow — didn't cover related entity types your master list also includes (Core Investment Companies, Standalone Primary Dealers, Mortgage Guarantee Companies, Non-Operative Financial Holding Companies, Housing Finance Companies).
- Never did the random-sample spot-check on `not_nbfc` and `nbfc_citation_unmatched` to get a real error rate — was proposed, not executed.

**8. Latest refinement: discard vs. general**
Moved from "no NBFC keyword → discard" to something sharper: check if the title explicitly names a *different*, specific RBI entity type (Regional Rural Banks, Urban Co-op Banks, etc. — pulled from RBI's own entity taxonomy) — that's a deliberate scope statement, a much stronger discard signal than mere absence of the word NBFC. Titles matching neither NBFC nor another named entity become `general` — genuinely ambiguous, handed to the model. Caveat flagged: a title naming *both* (e.g. a Commercial Banks direction cross-referenced within an NBFC amendment) must not be discarded — logic needs to be "other entity named AND NBFC not also named." Also flagged: worth spot-checking the `discard` bucket to make sure no bank/co-op-titled circular quietly carries an NBFC-specific clause in its body.

In [9]:
# ============================================================
# Cell 1: load data
# ============================================================
import re
import pandas as pd
import os
from collections import defaultdict
from difflib import SequenceMatcher

notebook_path = os.getcwd()
in_dir_notifications_raw = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "notification_historical", "notification.jsonl"))
in_dir_master_direction   = os.path.abspath(os.path.join(notebook_path, "..", "..", "Data", "master_directory", "master_directory.jsonl"))

circulars         = pd.read_json(in_dir_notifications_raw, lines=True)
master_directions = pd.read_json(in_dir_master_direction, lines=True)
print(f"circulars: {len(circulars)}, master_directions: {len(master_directions)}")


# ============================================================
# Cell 2: citation regex + normalizer
# ============================================================
DOC_TYPE = r"(?:Directions?|Guidelines?|Regulations?|Rules?|Circulars?|Framework|Scheme)"

p1 = re.compile(
    r"Reserve Bank of India\s*[-–—]?\s*\(([^)]+)\)"
    r"(?:\s*\([^)]*\))*"
    r"\s*(?:[A-Za-z]+\s+){0,3}" + DOC_TYPE,
    re.IGNORECASE
)
p2 = re.compile(r"Reserve Bank of India\s*[-–—]\s*([A-Za-z ,]+?),?\s*Directions", re.IGNORECASE)

norm = lambda s: re.sub(r"\s+", " ", s.replace("–", "-").replace("—", "-")).strip(" ,-").lower()

def extract_names(text):
    if not isinstance(text, str):
        return []
    return [norm(x) for x in (p1.findall(text) + p2.findall(text))]


# ============================================================
# Cell 3: title + full-text + "lead paragraph" extraction
# ============================================================
# Lead cutoff (start of numbered paragraph "2.") is ONLY for the "does this
# doc even carry a citation" bucketing check further down. Matching itself
# always runs on found_text (full text) -- restricting matching to the lead
# was a real bug that cut real matches from 102 to 79.
lead_break = re.compile(r"\n\s*2\.\s")

def get_lead(text):
    if not isinstance(text, str):
        return ""
    m = lead_break.search(text)
    return text[:m.start()] if m else text[:600]

circulars["found_title"]     = circulars["title"].apply(extract_names)                        # title citations
circulars["found_text"]      = circulars["text"].apply(extract_names)                         # full text -- used for matching
circulars["found_text_lead"] = circulars["text"].apply(lambda t: extract_names(get_lead(t)))  # lead only -- used for bucketing


# ============================================================
# Cell 4: build master_lookup, flagging collisions instead of silently overwriting
# ============================================================
master_directions["extracted_names"] = master_directions["title"].apply(extract_names)

name_to_ids = defaultdict(set)
for _, r in master_directions.iterrows():
    for name in r["extracted_names"]:
        name_to_ids[name].add(r["id"])

collisions = {k: v for k, v in name_to_ids.items() if len(v) > 1}
print(f"{len(collisions)} names collide across master_directions (excluded from lookup):")
print(collisions)

unlinkable = master_directions[master_directions["extracted_names"].str.len() == 0]
print(f"{len(unlinkable)}/{len(master_directions)} master directions have no extractable name (can never be matched to)")

master_lookup = {name: next(iter(ids)) for name, ids in name_to_ids.items() if len(ids) == 1}
master_keys = list(master_lookup.keys())


# ============================================================
# Cell 5 (moved up, was Cell 7): NBFC relevance + discard signal --
# computed BEFORE matching, so the title/text lookup against
# master_lookup is only attempted on circulars that could plausibly be
# NBFC-relevant. A circular whose title names a different specific RBI
# entity (Commercial Banks, Co-op Banks, etc.) and does NOT also name
# NBFC is discarded here, up front -- so it can never get spuriously
# "linked" just because its body cites an NBFC master direction in passing.
# ============================================================
nbfc_pattern = re.compile(
    r"non[\s-]?banking financial compan|nbfc"
    r"|core investment compan(?:y|ies)|\bcic\b"
    r"|standalone primary dealer|\bspd\b"
    r"|mortgage guarantee compan(?:y|ies)|\bmgc\b"
    r"|non-?operative financial holding compan(?:y|ies)|\bnofhc\b"
    r"|housing finance compan(?:y|ies)|\bhfc\b",
    re.IGNORECASE
)
circulars["is_nbfc_relevant"] = (
    circulars["text"].str.contains(nbfc_pattern, na=False) |
    circulars["title"].str.contains(nbfc_pattern, na=False)
)

# Entities RBI regulates that are explicitly NOT NBFC-family. Checked on
# title only -- that's where scope is declared.
other_entity_pattern = re.compile(
    r"regional rural bank"
    r"|urban co-?operative bank"
    r"|state co-?operative bank"
    r"|district central co-?operative bank"
    r"|scheduled commercial bank"
    r"|commercial bank"
    r"|payments? bank"
    r"|small finance bank"
    r"|local area bank"
    r"|co-?operative bank",
    re.IGNORECASE
)
circulars["names_other_entity"] = circulars["title"].str.contains(other_entity_pattern, na=False)

# Discard = names a different specific entity AND does not also name NBFC.
# (A title naming both -- e.g. a Commercial Banks direction cross-referenced
# inside an NBFC amendment -- must NOT be discarded, hence the ~is_nbfc_relevant.)
circulars["discard_pre_match"] = circulars["names_other_entity"] & ~circulars["is_nbfc_relevant"]

print(f"{circulars['discard_pre_match'].sum()} / {len(circulars)} circulars discarded "
      f"before matching (named a different entity, not NBFC)")


# ============================================================
# Cell 6 (was Cell 5): title/full-text matching -- skips anything already
# discarded, so those rows never get a chance at a spurious title/text match.
# ============================================================
def match_row(row):
    if row["discard_pre_match"]:
        return None, "discarded"
    for n in row["found_title"]:
        if n in master_lookup:
            return master_lookup[n], "title_match"
    for n in row["found_text"]:          # full text, not lead
        if n in master_lookup:
            return master_lookup[n], "text_match"
    return None, "no_match"

circulars[["matched_id", "match_method"]] = circulars.apply(lambda r: pd.Series(match_row(r)), axis=1)
print(circulars["match_method"].value_counts())

recovered_by_full_text = circulars[
    (circulars["match_method"].isin(["title_match", "text_match"])) &   # excludes "discarded" too
    (circulars["found_title"].str.len() == 0) &
    (~circulars["found_text_lead"].apply(lambda lst: any(n in master_lookup for n in lst)))
]
print(f"{len(recovered_by_full_text)} matches only found past the lead cutoff")
recovered_by_full_text[["id", "title", "matched_id"]]


# ============================================================
# Cell 7 (was Cell 6): fuzzy candidates -- unchanged. Naturally only runs
# on match_method == "no_match" rows, so discarded rows are excluded here
# for free (they're labeled "discarded", not "no_match").
# ============================================================
def char_diff(a, b):
    diff = 0
    for tag, i1, i2, j1, j2 in SequenceMatcher(None, a, b).get_opcodes():
        if tag != "equal":
            diff += max(i2 - i1, j2 - j1)
    return diff

fuzzy_candidates = []
for idx, row in circulars[circulars["match_method"] == "no_match"].iterrows():
    for n in (row["found_title"] + row["found_text_lead"]):
        for key in master_keys:
            if char_diff(n, key) <= 5:
                fuzzy_candidates.append((idx, row["id"], n, key))
                break

fuzzy_df = pd.DataFrame(fuzzy_candidates, columns=["row_idx", "circular_id", "extracted_name", "closest_master_name"])
print(f"{len(fuzzy_df)} candidates")
fuzzy_df


# ============================================================
# Cell 7b (was Cell 6b): apply fuzzy matches -- unchanged.
# ============================================================
dupe_check = fuzzy_df.groupby("row_idx")["closest_master_name"].apply(
    lambda names: len({master_lookup[n] for n in names})
)
ambiguous_rows = dupe_check[dupe_check > 1].index
if len(ambiguous_rows):
    print(f"{len(ambiguous_rows)} rows have conflicting fuzzy candidates -- review before applying:")
    print(fuzzy_df[fuzzy_df["row_idx"].isin(ambiguous_rows)])

safe_fuzzy = fuzzy_df[~fuzzy_df["row_idx"].isin(ambiguous_rows)].drop_duplicates("row_idx")
for _, r in safe_fuzzy.iterrows():
    circulars.loc[r["row_idx"], "matched_id"]   = master_lookup[r["closest_master_name"]]
    circulars.loc[r["row_idx"], "match_method"] = "fuzzy_match"

print(circulars["match_method"].value_counts())


# ============================================================
# Cell 8 (was Cell 7, simplified): bucket assignment. discard is already
# decided pre-match, so this just reads match_method off directly instead
# of re-deriving it from is_nbfc_relevant / names_other_entity.
# ============================================================
def bucket(row):
    if row["match_method"] == "discarded":
        return "discard"
    if row["match_method"] != "no_match":
        return "linked"
    if row["is_nbfc_relevant"]:
        has_citation = len(row["found_title"]) > 0 or len(row["found_text_lead"]) > 0
        return "nbfc_citation_unmatched" if has_citation else "nbfc_standalone"
    return "general"   # not discarded, not NBFC-named, not matched -- genuinely ambiguous

circulars["bucket"] = circulars.apply(bucket, axis=1)
print(circulars["bucket"].value_counts())


# ============================================================
# Cell 9 (was Cell 8): subject_code cross-check -- filter tightened to the
# real match methods, so a "discarded" row (matched_id=None) never gets
# compared against a coincidental subject_code hit and flagged as a
# disagreement.
# ============================================================
ref_pattern = re.compile(r"([A-Z]+(?:\.[A-Z]+)+\.\d+)/([\d-]+)/(\d{4}-\d{2})")
circulars["subject_code"]         = circulars["text"].str.extract(ref_pattern)[1]
master_directions["subject_code"] = master_directions["text"].str.extract(ref_pattern)[1]

code_counts = master_directions.dropna(subset=["subject_code"]).groupby("subject_code")["id"].nunique()
generic_codes = code_counts[code_counts > 1].index.tolist()
clean_master = master_directions[~master_directions["subject_code"].isin(generic_codes)]

code_matches = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code"]],
    on="subject_code", how="left", suffixes=("", "_master")
)
both = code_matches[
    code_matches["match_method"].isin(["title_match", "text_match", "fuzzy_match"]) &
    code_matches["id_master"].notna()
]
disagreements = both[both["matched_id"] != both["id_master"]]
print(f"{len(disagreements)} / {len(both)} disagree between regex-match and code-match")


# ============================================================
# Cell 10 (was Cell 9): final tally -- unchanged logic.
# ============================================================
print(circulars["bucket"].value_counts())
nbfc_total = circulars["bucket"].isin(["linked", "nbfc_citation_unmatched", "nbfc_standalone"]).sum()
print(f"NBFC-relevant: {nbfc_total} / {len(circulars)}")
print(f"Discard (named a different specific entity): {(circulars['bucket']=='discard').sum()} / {len(circulars)}")
print(f"General (ambiguous, needs model review): {(circulars['bucket']=='general').sum()} / {len(circulars)}")

circulars: 766, master_directions: 44
1 names collide across master_directions (excluded from lookup):
{'non-banking financial companies - miscellaneous': {13586, 12931}}
0/44 master directions have no extractable name (can never be matched to)
395 / 766 circulars discarded before matching (named a different entity, not NBFC)
match_method
discarded      395
no_match       270
title_match     76
text_match      25
Name: count, dtype: int64
1 matches only found past the lead cutoff
3 candidates
match_method
discarded      395
no_match       268
title_match     76
text_match      25
fuzzy_match      2
Name: count, dtype: int64
bucket
discard                    395
nbfc_citation_unmatched    154
general                    113
linked                     103
nbfc_standalone              1
Name: count, dtype: int64
0 / 7 disagree between regex-match and code-match
bucket
discard                    395
nbfc_citation_unmatched    154
general                    113
linked                     103

In [10]:
pd.set_option('display.max_colwidth', None)
def review_sample(method, n=15, seed=0):
    pool = circulars[circulars["match_method"] == method]
    sample = pool.sample(min(n, len(pool)), random_state=seed).copy()
    return sample.merge(
        master_directions[["id", "title"]].rename(columns={"title": "master_title"}),
        left_on="matched_id", right_on="id", suffixes=("", "_m")
    )[["id", "title", "master_title", "match_method"]]

review_sample("title_match")
review_sample("text_match")
review_sample("fuzzy_match")   # review ALL of these, not just a sample, if the count is small

,id,title,master_title,match_method
0,13379,"Reserve Bank of India (Non-Banking Financial Companies– Undertaking of Financial Services) –Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Undertaking of Financial Services) Directions, 2025 (Updated as on July 01, 2026)",fuzzy_match
1,13213,"Reserve Bank of India (Non-Operative Financial Holding Company) (Amendment) Directions, 2025","Reserve Bank of India (Non-Operative Financial Holding Companies) Directions, 2025 (Updated as on December 05, 2025)",fuzzy_match


In [11]:
code_lookup = circulars.merge(
    clean_master.dropna(subset=["subject_code"])[["id", "subject_code", "title"]],
    on="subject_code", how="left", suffixes=("", "_code_master")
)

recoverable = code_lookup[
    (code_lookup["match_method"] == "no_match") &
    code_lookup["id_code_master"].notna()
]
print(f"{len(recoverable)} unmatched circulars have a subject_code hit against a master direction")
recoverable[["id", "title", "title_code_master"]]

7 unmatched circulars have a subject_code hit against a master direction


,id,title,title_code_master
58,12983,"Reserve Bank of India (All India Financial Institutions - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on June 16, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
155,13080,"Reserve Bank of India (Local Area Banks – Prudential Norms on Capital Adequacy) Directions, 2025","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
172,13097,"Reserve Bank of India (Payments Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (updated as on May 08, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
202,13127,"Reserve Bank of India (Small Finance Banks – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
234,13159,"Reserve Bank of India (Commercial Banks - Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
342,13267,"Reserve Bank of India (All India Financial Institutions (AIFIs) - Prudential Norms on Capital Adequacy) Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
579,13504,"Reserve Bank of India (All India Financial Institutions – Prudential Norms on Capital Adequacy) Third Amendment Directions, 2026","Reserve Bank of India (Non-Banking Financial Companies – Prudential Norms on Capital Adequacy) Directions, 2025 (Updated as on July 01, 2026)"
